In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
dbutils.widgets.removeAll()

In [0]:
dbutils.widgets.text("catalogo", "bank_dev")
dbutils.widgets.text("esquema_bronze", "bronze")
dbutils.widgets.text("esquema_silver", "silver")
dbutils.widgets.text("storageName", "saprojectbankdeveastus")
dbutils.widgets.text("containerSilver", "silver")

In [0]:
catalogo         = dbutils.widgets.get("catalogo")
esquema_bronze   = dbutils.widgets.get("esquema_bronze")
esquema_silver   = dbutils.widgets.get("esquema_silver")
storageName      = dbutils.widgets.get("storageName")
containerSilver  = dbutils.widgets.get("containerSilver")

In [0]:

tabla_bronze_mcc_codes = f"{catalogo}.{esquema_bronze}.mcc_codes"
tabla_silver_mcc_codes = f"{catalogo}.{esquema_silver}.dim_mcc_codes"

In [0]:
_df = spark.read.table(tabla_bronze_mcc_codes)

In [0]:
# ===================== TRANSFORMACIÓN Y CARGA MCC (STATIC OVERWRITE) =====================

# 1. Transformación
df_silver_mcc = _df \
    .withColumnRenamed("mcc", "mcc_code") \
    .withColumnRenamed("description", "mcc_description") \
    .withColumn("silver_load_date", F.current_timestamp()) \
    .drop("ingestion_date") # Borramos la de bronze para no duplicar fechas

# 2. Guardado en Silver (Modo Overwrite)
print(f"Sobrescribiendo la tabla catálogo {tabla_silver_mcc_codes}...")

(df_silver_mcc.write
 .format("delta")
 .mode("overwrite") # Borra lo viejo y pone lo nuevo, ideal para catálogos
 .saveAsTable(tabla_silver_mcc_codes))

print("Carga de MCC completada exitosamente.")